In [121]:
import os
import PyPDF2
import numpy as np
import pandas as pd
from tabula.io import read_pdf
from datetime import datetime
import re

In [122]:
file_name = r"C:\Users\admin\Downloads\13.06.2024 £288.82 Battery Dynamics.pdf"

r"C:\Users\admin\Downloads\02.01.2024 £88.76 Battery Dynamics.pdf"

'C:\\Users\\admin\\Downloads\\02.01.2024 £88.76 Battery Dynamics.pdf'

In [123]:
invoice_type = "Products"

input_file = fr"C:\Users\admin\Downloads\24.08.2023 £48.07 Battery Dynamics.pdf"

In [124]:
name = "Battery Dynamics"

box1 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(0,462,40,594),
                  columns=[594],
                  pandas_options={'header': None},
                  encoding="windows-1254")

heading1 = box1[0]
display(heading1)

docnum = str(heading1[0][0])
print(docnum)


,0
0,23701401


23701401


In [125]:
table1 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(202,10,253,580),
                  columns=[71,162,261,361,451,580],
                  pandas_options={'header': None},
                  encoding="windows-1254")
heading2 = table1[0]
display(heading2)

column_date = heading2.columns[heading2.iloc[0] == "Invoice Date"].tolist()[0]
date = heading2[column_date][1]
date = str(datetime.strptime(date, "%d/%m/%Y"))
print(date)

column_ordernum = heading2.columns[heading2.iloc[0] == "Purchase Order No."].tolist()[0]
ordernum = heading2[column_ordernum][1]

transfernum = None
print(ordernum)
print(transfernum)

,0,1,2,3,4,5
0,Account No.,Invoice Date,Purchase Order No.,Sales Order No.,Delivery Note No.,Account Manager
1,152739,24/08/2023,PO24768,501193,7575,Tim Jones


2023-08-24 00:00:00
PO24768
None


In [126]:
with open(input_file,'rb') as pdf_file:
    pdf_reader = PyPDF2.PdfReader(pdf_file)
    num_pages = len(pdf_reader.pages)

print(num_pages)

1


In [127]:
#create loop to go through the pages
all_content = []
for page in range(1, num_pages + 1):
    table2 = read_pdf(input_file,
                    pages=page,
                    silent=True,
                    guess=False,
                    area=(270,10,590,580),
                    columns=[50,120,376,470,518,580],
                    pandas_options={'header': None},
                    encoding="windows-1254")
    contenti = table2[0]
    all_content.append(contenti)

content = pd.concat(all_content).reset_index(drop=True)
display(content)

,0,1,2,3,4,5
0,400044,12LS-4.5,"Q-Batteries 12LS-4.5 12V 4,5Ah AGM VRLA Standb...",4,7.89,31.56


In [128]:
content.rename(columns={
    0: 'Item',
    1: 'Code',
    2: 'Description',
    3: 'Quantity',
    4: 'Unit Price',
    5: 'Net Amount'}, inplace=True)

display(content)

,Item,Code,Description,Quantity,Unit Price,Net Amount
0,400044,12LS-4.5,"Q-Batteries 12LS-4.5 12V 4,5Ah AGM VRLA Standb...",4,7.89,31.56


In [129]:
dict_content = content.to_dict(orient='records')
dict_content

line_items=[]
for item in dict_content:

    partNum = item['Item']
    desc = item['Description']
    quantity = item['Quantity']
    netTotal = item['Net Amount']

    print(partNum)
    
    line_item = {"line_type": "inventory",
                        "sku": str(partNum),
                        "name": desc,
                        "quantity": int(quantity),
                        "net_total": float(netTotal),
                        "tax_type": "INPUT2"}
    
    line_items.append(line_item)
    
print(line_items)

400044
[{'line_type': 'inventory', 'sku': '400044', 'name': 'Q-Batteries 12LS-4.5 12V 4,5Ah AGM VRLA Standby battery', 'quantity': 4, 'net_total': 31.56, 'tax_type': 'INPUT2'}]


In [130]:
table3 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(583,420,652,571),
                  columns=[530,571],
                  pandas_options={'header': None},
                  encoding='windows-1254')

total_content=table3[0]
display(total_content)

row_index = total_content.index[total_content[0] == "Total Amount: GBP"].tolist()[0]
final_total = float(str(total_content[1][row_index]).replace(',',''))
display(final_total)

row_shipping = total_content.index[total_content[0] == "Delivery: GBP"].tolist()[0]
Shipping = float(str(total_content[1][row_shipping]).replace(',',''))
print(Shipping)



,0,1
0,Total Amount Ex VAT: GBP,31.56
1,Delivery: GBP,8.50
2,Total VAT 20%: GBP,8.01
3,Total Amount: GBP,48.07


48.07

8.5


In [131]:
#Adding shipping to the line_items
line_shipping = {"line_type":"shipping_expense",
            "sku": None,
            "name": "shipping",
            "Quantity": int(1),
            "net_total": float(Shipping),
            "tax_type": "INPUT2"}

line_items.append(line_shipping)
print(line_items)

[{'line_type': 'inventory', 'sku': '400044', 'name': 'Q-Batteries 12LS-4.5 12V 4,5Ah AGM VRLA Standby battery', 'quantity': 4, 'net_total': 31.56, 'tax_type': 'INPUT2'}, {'line_type': 'Postage, Freight & Courier', 'sku': None, 'name': 'shipping', 'Quantity': 1, 'net_total': 8.5, 'tax_type': 'INPUT2'}]


In [132]:
payload = {}
keys = ["Source File",
        "Type",
        "Name",
        "Date",
        "Reference No.",
        "Order No.",
        "Transfer No.",
        "Document No.",
        "Line Items",
        "Total"]

values = [file_name,
        invoice_type,
        name,
        date,
        docnum,
        ordernum,
        transfernum,
        None,
        line_items,
        final_total]

for i, key in enumerate(keys):
    payload[key] = values[i]

payload

{'Source File': 'C:\\Users\\admin\\Downloads\\13.06.2024 £288.82 Battery Dynamics.pdf',
 'Type': 'Products',
 'Name': 'Battery Dynamics',
 'Date': '2023-08-24 00:00:00',
 'Reference No.': '23701401',
 'Order No.': 'PO24768',
 'Transfer No.': None,
 'Document No.': None,
 'Line Items': [{'line_type': 'inventory',
   'sku': '400044',
   'name': 'Q-Batteries 12LS-4.5 12V 4,5Ah AGM VRLA Standby battery',
   'quantity': 4,
   'net_total': 31.56,
   'tax_type': 'INPUT2'},
  {'line_type': 'Postage, Freight & Courier',
   'sku': None,
   'name': 'shipping',
   'Quantity': 1,
   'net_total': 8.5,
   'tax_type': 'INPUT2'}],
 'Total': 48.07}